In [0]:
%sql

CREATE OR REPLACE TEMP VIEW stg_journal AS
SELECT
    journal_id,
    journal_name,
    type,
    code,
    created_at,
    updated_at,
    is_updated,
    is_deleted
FROM bakery_pos_develop.silver.account_journal
QUALIFY ROW_NUMBER() OVER(
    PARTITION BY journal_id
    ORDER BY updated_at DESC
) = 1

In [0]:
%sql

CREATE TABLE IF NOT EXISTS bakery_pos_develop.gold.dim_account_journal (
    journal_sk BIGINT GENERATED ALWAYS AS IDENTITY,
    journal_id INT,
    journal_name STRING,
    type STRING,
    code STRING,
    created_at TIMESTAMP,
    updated_at TIMESTAMP,
    is_updated BOOLEAN,
    is_deleted BOOLEAN,
    PRIMARY KEY (journal_sk)
)

In [0]:
%sql
MERGE INTO bakery_pos_develop.gold.dim_account_journal AS tgt
USING stg_journal AS src
ON tgt.journal_id = src.journal_id

WHEN MATCHED AND src.updated_at > tgt.updated_at THEN
    UPDATE SET
        tgt.journal_name = src.journal_name,
        tgt.type = src.type,
        tgt.code = src.code,
        tgt.created_at = src.created_at,
        tgt.updated_at = src.updated_at,
        tgt.is_updated = src.is_updated,
        tgt.is_deleted = src.is_deleted

WHEN NOT MATCHED THEN
    INSERT (
        journal_id,
        journal_name,
        type,
        code,
        created_at,
        updated_at,
        is_updated,
        is_deleted
    )
    VALUES (
        src.journal_id,
        src.journal_name,
        src.type,
        src.code,
        src.created_at,
        src.updated_at,
        src.is_updated,
        src.is_deleted
    )